# RAGognize 数据准备

本 Notebook 是 RAGognize 数据适配的可读入口。

**核心逻辑在 `src/ragognize_adapter/` 中，不在此 Notebook 中。**

In [ ]:
# 环境设置
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import sys
sys.path.insert(0, 'src')

from src.ragognize_adapter import (
    load_ragognize_dataset,
    create_prompt_split,
    apply_split,
    SOURCE_MODELS,
    UnifiedSample,
)

print(f"Python: {sys.executable}")
print(f"Models: {SOURCE_MODELS}")

## 1. 加载数据

In [ ]:
# 加载 RAGognize 数据集
dataset = load_ragognize_dataset()
print(f"Dataset loaded: {dataset}")

## 2. 创建 Split

In [ ]:
# 创建 prompt 级别的 train/validation split
# 同一 user_prompt_index 的 4 个模型回答会在同一 split
split_info = create_prompt_split(dataset, val_ratio=0.15, seed=42)

print(f"Train prompts: {split_info['train_count']}")
print(f"Val prompts: {split_info['val_count']}")
print(f"Test prompts: {split_info['test_count']}")

## 3. 展开为 UnifiedSamples

In [ ]:
# 展开为 UnifiedSample 列表
expanded = apply_split(dataset, split_info)

print(f"Train samples: {len(expanded['train'])}")
print(f"Val samples: {len(expanded['val'])}")
print(f"Test samples: {len(expanded['test'])}")

## 4. 样本预览

In [ ]:
def truncate(text, max_len=80):
    """截断文本用于显示"""
    if len(text) > max_len:
        return text[:max_len] + '...'
    return text

# 找一个 faithful 样本
faithful_sample = None
unfaithful_sample = None

for s in expanded['train']:
    if faithful_sample is None and s.faithfulness_label == 1:
        faithful_sample = s
    if unfaithful_sample is None and s.faithfulness_label == 0:
        unfaithful_sample = s
    if faithful_sample and unfaithful_sample:
        break

print("=" * 60)
print("FAITHFUL SAMPLE:")
print("=" * 60)
if faithful_sample:
    print(f"case_id: {faithful_sample.case_id}")
    print(f"question: {truncate(faithful_sample.question)}")
    print(f"answer: {truncate(faithful_sample.answer)}")
    print(f"faithfulness_label: {faithful_sample.faithfulness_label}")
    print(f"has_hallucination: {faithful_sample.has_hallucination}")
    print(f"spans: {len(faithful_sample.hallucination_spans)}")

print()
print("=" * 60)
print("UNFAITHFUL SAMPLE:")
print("=" * 60)
if unfaithful_sample:
    print(f"case_id: {unfaithful_sample.case_id}")
    print(f"question: {truncate(unfaithful_sample.question)}")
    print(f"answer: {truncate(unfaithful_sample.answer)}")
    print(f"faithfulness_label: {unfaithful_sample.faithfulness_label}")
    print(f"has_hallucination: {unfaithful_sample.has_hallucination}")
    print(f"spans: {len(unfaithful_sample.hallucination_spans)}")
    if unfaithful_sample.hallucination_spans:
        span = unfaithful_sample.hallucination_spans[0]
        print(f"span[0]: [{span.start}:{span.end}] '{truncate(span.text)}'")

## 5. 标签分布

In [ ]:
from collections import Counter

print("Faithfulness Label Distribution:")
print("-" * 40)
for split in ['train', 'val', 'test']:
    faithful = sum(1 for s in expanded[split] if s.faithfulness_label == 1)
    unfaithful = sum(1 for s in expanded[split] if s.faithfulness_label == 0)
    total = len(expanded[split])
    rate = faithful / total * 100 if total > 0 else 0
    print(f"{split:>6}: faithful={faithful:>5}, unfaithful={unfaithful:>5}, rate={rate:.1f}%")

## 6. 模型分布

In [ ]:
print("Model Distribution:")
print("-" * 40)
for split in ['train', 'val', 'test']:
    models = Counter(s.source_model for s in expanded[split])
    print(f"{split:>6}: {dict(models)}")

## 7. 访问更多样本

In [ ]:
# 访问样本的属性
sample = expanded['train'][0]
print(f"UnifiedSample attributes:")
print(f"  case_id: {sample.case_id}")
print(f"  user_prompt_index: {sample.user_prompt_index}")
print(f"  question: {truncate(sample.question)}")
print(f"  answer: {truncate(sample.answer)}")
print(f"  chunks: {len(sample.chunks)} chunks")
print(f"  chunk_titles: {len(sample.chunk_titles)} titles")
print(f"  golden_answer: {truncate(sample.golden_answer)}")
print(f"  hallucination_spans: {len(sample.hallucination_spans)} spans")
print(f"  has_hallucination: {sample.has_hallucination}")
print(f"  faithfulness_label: {sample.faithfulness_label}")
print(f"  answerable: {sample.answerable}")
print(f"  source_model: {sample.source_model}")
print(f"  information_type: {sample.information_type}")
print(f"  category: {sample.category}")

## 8. 转换为字典格式

In [ ]:
# 转换为字典格式便于序列化
sample_dict = expanded['train'][0].to_dict()
print("Sample as dict:")
for key, value in sample_dict.items():
    if isinstance(value, (list, str)) and len(str(value)) > 50:
        print(f"  {key}: {type(value).__name__}, len={len(value)}")
    else:
        print(f"  {key}: {value}")

---

**核心逻辑**: 参见 `src/ragognize_adapter/` 模块